In [1]:
Sys.setenv(RETICULATE_PYTHON = "/usr/bin/python3")

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(reticulate)
  library(CellEnergy)
})

cat("[INFO] Python used by reticulate:\n")
print(reticulate::py_config())

cat("[INFO] Module check:\n")
cat("pandas:", reticulate::py_module_available("pandas"), "\n")
cat("numpy :", reticulate::py_module_available("numpy"), "\n")
cat("scipy :", reticulate::py_module_available("scipy"), "\n")

Warning message:
“no DISPLAY variable so Tk is not available”


[INFO] Python used by reticulate:
python:         /usr/bin/python3
libpython:      /usr/lib/python3.10/config-3.10-x86_64-linux-gnu/libpython3.10.so
pythonhome:     //usr://usr
version:        3.10.12 (main, Jan 17 2025, 14:35:34) [GCC 11.4.0]
numpy:          /usr/lib/python3/dist-packages/numpy
numpy_version:  1.21.5

NOTE: Python version was forced by RETICULATE_PYTHON
[INFO] Module check:
pandas: TRUE 
numpy : TRUE 
scipy : TRUE 


In [ ]:


suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(CellEnergy)
})



data_dir <- "/path/pbmc3k/filtered_gene_bc_matrices/hg19/"
out_dir <- "/path/PBMC3k/Seurat_CellEnergy_full_pipeline"

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

min_cells <- 3
min_features <- 200
max_features <- 2500
max_mt <- 5
n_hvg <- 2000

cat("[INFO] Input directory:\n")
cat(data_dir, "\n\n")

cat("[INFO] Output directory:\n")
cat(out_dir, "\n\n")

if (!dir.exists(data_dir)) {
  stop("[ERROR] data_dir does not exist: ", data_dir)
}


write_matrix_with_id <- function(mat, file, id_col = "ID") {
  mat <- as.matrix(mat)
  df <- as.data.frame(mat, check.names = FALSE)
  df <- cbind(setNames(data.frame(rownames(df), check.names = FALSE), id_col), df)
  data.table::fwrite(df, file = file)
}

minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)

  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min

  col_range[col_range == 0 | is.na(col_range)] <- 1

  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  scaled[is.na(scaled)] <- 0

  return(scaled)
}

get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "counts", slot_name = "counts") {
  DefaultAssay(obj) <- assay

  tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) {
      GetAssayData(obj, assay = assay, slot = slot_name)
    }
  )
}


cat("[INFO] Reading PBMC3k 10X matrix...\n")

pbmc.data <- Read10X(data.dir = data_dir)

cat("[INFO] Raw matrix dimension, genes x cells:\n")
print(dim(pbmc.data))

pbmc <- CreateSeuratObject(
  counts = pbmc.data,
  project = "PBMC3k",
  min.cells = min_cells,
  min.features = min_features
)

cat("[INFO] Seurat object before QC:\n")
print(pbmc)

pbmc[["percent.mt"]] <- PercentageFeatureSet(pbmc, pattern = "^MT-")

qc_before <- data.frame(
  n_cells_before_qc = ncol(pbmc),
  n_genes_before_qc = nrow(pbmc)
)

pbmc <- subset(
  pbmc,
  subset = nFeature_RNA > min_features &
    nFeature_RNA < max_features &
    percent.mt < max_mt
)

qc_after <- data.frame(
  n_cells_after_qc = ncol(pbmc),
  n_genes_after_qc = nrow(pbmc),
  min_features = min_features,
  max_features = max_features,
  max_mt = max_mt
)

qc_summary <- cbind(qc_before, qc_after)

data.table::fwrite(
  qc_summary,
  file.path(out_dir, "QC_summary.csv")
)

cat("[INFO] Seurat object after QC:\n")
print(pbmc)


pbmc <- NormalizeData(
  pbmc,
  normalization.method = "LogNormalize",
  scale.factor = 10000
)

pbmc <- FindVariableFeatures(
  pbmc,
  selection.method = "vst",
  nfeatures = min(n_hvg, nrow(pbmc))
)

hvg_genes <- VariableFeatures(pbmc)

cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

data.table::fwrite(
  data.frame(HVG = hvg_genes),
  file.path(out_dir, paste0("HVG_", length(hvg_genes), "_gene_list.csv"))
)

pbmc <- ScaleData(
  pbmc,
  features = rownames(pbmc)
)


pbmc <- RunPCA(
  pbmc,
  features = hvg_genes,
  verbose = FALSE
)

pbmc <- FindNeighbors(
  pbmc,
  dims = 1:10
)

pbmc <- FindClusters(
  pbmc,
  resolution = 0.5
)

cat("[INFO] Seurat cluster distribution:\n")
print(table(pbmc$seurat_clusters))

cluster_to_celltype <- c(
  "0" = "Naive_CD4_T",
  "1" = "CD14_Monocyte",
  "2" = "Memory_CD4_T",
  "3" = "B_cell",
  "4" = "CD8_T",
  "5" = "FCGR3A_Monocyte",
  "6" = "NK_cell",
  "7" = "Dendritic_cell",
  "8" = "Platelet"
)

cluster_chr <- as.character(pbmc$seurat_clusters)

celltype <- cluster_to_celltype[cluster_chr]

celltype <- unname(celltype)

missing_idx <- is.na(celltype)
celltype[missing_idx] <- paste0("Cluster_", cluster_chr[missing_idx])

stopifnot(length(celltype) == ncol(pbmc))

pbmc <- AddMetaData(
  object = pbmc,
  metadata = celltype,
  col.name = "celltype"
)

cat("[INFO] Cell type label distribution:\n")
print(table(pbmc$celltype))

cell_labels_after_qc <- data.frame(
  Cell = colnames(pbmc),
  Label = pbmc$celltype,
  Cluster = as.character(pbmc$seurat_clusters),
  stringsAsFactors = FALSE
)

data.table::fwrite(
  cell_labels_after_qc,
  file.path(out_dir, "cell_labels_after_QC.csv")
)

metadata_after_qc <- data.frame(
  Cell = colnames(pbmc),
  nCount_RNA = pbmc$nCount_RNA,
  nFeature_RNA = pbmc$nFeature_RNA,
  percent.mt = pbmc$percent.mt,
  Cluster = as.character(pbmc$seurat_clusters),
  Label = pbmc$celltype,
  stringsAsFactors = FALSE
)

data.table::fwrite(
  metadata_after_qc,
  file.path(out_dir, "metadata_after_QC.csv")
)


raw_hvg_counts_gene_by_cell <- get_assay_data_compat(
  pbmc,
  assay = "RNA",
  layer_name = "counts",
  slot_name = "counts"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  pbmc,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

raw_hvg_cell_by_gene <- t(as.matrix(raw_hvg_counts_gene_by_cell))
raw_hvg_minmax <- minmax_scale_columns(raw_hvg_cell_by_gene)

write_matrix_with_id(
  raw_hvg_cell_by_gene,
  file.path(out_dir, "final_raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  raw_hvg_minmax,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 1 saved:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n\n")


scaled_for_cellenergy_path <- file.path(
  out_dir,
  "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv"
)

write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_for_cellenergy_path,
  quote = FALSE,
  row.names = TRUE
)

cat("[INFO] Running CellEnergy::calcGEn...\n")

result <- CellEnergy::calcGEn(
  scaled_for_cellenergy_path,
  verbose = TRUE
)

GLNE <- result$GLNE
GLNE_mat <- as.matrix(GLNE)

write_matrix_with_id(
  GLNE_mat,
  file.path(out_dir, "GLNE_raw_output.csv"),
  id_col = "ID"
)

glne_rows_are_cells <- sum(rownames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))
glne_cols_are_cells <- sum(colnames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))

if (glne_rows_are_cells > glne_cols_are_cells) {
  cat("[INFO] GLNE appears to be cells x genes.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing.\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells <- intersect(rownames(raw_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes <- intersect(colnames(raw_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between View1 and GLNE:\n")
print(length(common_cells))

cat("[INFO] Common genes between View1 and GLNE:\n")
print(length(common_genes))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cells between View1 and GLNE.")
}

if (length(common_genes) == 0) {
  stop("[ERROR] No common genes between View1 and GLNE.")
}

raw_hvg_cell_by_gene_final <- raw_hvg_cell_by_gene[common_cells, common_genes, drop = FALSE]
GLNE_cell_by_gene <- GLNE_cell_by_gene[common_cells, common_genes, drop = FALSE]

raw_hvg_minmax_final <- minmax_scale_columns(raw_hvg_cell_by_gene_final)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

write_matrix_with_id(
  raw_hvg_cell_by_gene_final,
  file.path(out_dir, "final_raw_HVG_counts_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  raw_hvg_minmax_final,
  file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_cell_by_gene,
  file.path(out_dir, "final_GLNE_cell_by_gene.csv"),
  id_col = "Cell"
)

write_matrix_with_id(
  GLNE_minmax,
  file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"),
  id_col = "Cell"
)

cat("[OK] View 2 saved:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n\n")


seurat_rds <- file.path(
  out_dir,
  "PBMC3k_Seurat_CellEnergy_processed_no_regression.rds"
)

saveRDS(pbmc, file = seurat_rds)

cat("\n[DONE] PBMC3k preprocessing finished.\n")
cat("1. Labels:\n")
cat(file.path(out_dir, "cell_labels_after_QC.csv"), "\n")
cat("2. View 1:\n")
cat(file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv"), "\n")
cat("3. View 2:\n")
cat(file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv"), "\n")
cat("4. Seurat RDS:\n")
cat(seurat_rds, "\n")

[INFO] Input directory:
/home/zhanghang/pbmc3k/filtered_gene_bc_matrices/hg19/ 

[INFO] Output directory:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline 

[INFO] Reading PBMC3k 10X matrix...
[INFO] Raw matrix dimension, genes x cells:
[1] 32738  2700


Warning message:
“Feature names cannot have underscores ('_'), replacing with dashes ('-')”


[INFO] Seurat object before QC:
An object of class Seurat 
13714 features across 2700 samples within 1 assay 
Active assay: RNA (13714 features, 0 variable features)
 1 layer present: counts
[INFO] Seurat object after QC:
An object of class Seurat 
13714 features across 2638 samples within 1 assay 
Active assay: RNA (13714 features, 0 variable features)
 1 layer present: counts


Normalizing layer: counts

Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000


Centering and scaling data matrix

Computing nearest neighbor graph

Computing SNN



Modularity Optimizer version 1.3.0 by Ludo Waltman and Nees Jan van Eck

Number of nodes: 2638
Number of edges: 95927

Running Louvain algorithm...
Maximum modularity in 10 random starts: 0.8728
Number of communities: 9
Elapsed time: 0 seconds
[INFO] Seurat cluster distribution:

  0   1   2   3   4   5   6   7   8 
684 481 476 344 291 162 155  32  13 
[INFO] Cell type label distribution:

         B_cell   CD14_Monocyte           CD8_T  Dendritic_cell FCGR3A_Monocyte 
            344             481             291              32             162 
   Memory_CD4_T         NK_cell     Naive_CD4_T        Platelet 
            476             155             684              13 
[OK] View 1 saved:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 

[INFO] Running CellEnergy::calcGEn...


2026-06-11 22:33:05.714797: Starting calculation.

2026-06-11 22:33:44.147373: Complete GLNE and cell Energy calculation.



[INFO] GLNE appears to be genes x cells. Transposing.
[INFO] Common cells between View1 and GLNE:
[1] 2638
[INFO] Common genes between View1 and GLNE:
[1] 2000
[OK] View 2 saved:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 


[DONE] PBMC3k preprocessing finished.
1. Labels:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/cell_labels_after_QC.csv 
2. View 1:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/final_raw_HVG_counts_MinMax_cell_by_gene.csv 
3. View 2:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/final_GLNE_MinMax_cell_by_gene.csv 
4. Seurat RDS:
/home/zhanghang/PBMC3k/Seurat_CellEnergy_full_pipeline/PBMC3k_Seurat_CellEnergy_processed_no_regression.rds 
